## Setup

In [3]:
!pip install -q trl bitsandbytes accelerate peft datasets transformers evaluate rouge_score


## Authentication

In [4]:
import os, torch
from huggingface_hub import login
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')


GPUs: 1
  GPU 0: Tesla T4  15.6 GB


## 1 · Dataset Preparation

> Loading `forget10` (400 samples) and `retain90` (3600 samples) from the TOFU dataset.

In [5]:
import os
from datasets import load_dataset, concatenate_datasets

DATA_DIR = '/content/data_splits'

print('[1] Loading TOFU datasets...')

forget_dataset = load_dataset('locuslab/TOFU', 'forget10', split='train')
retain_dataset = load_dataset('locuslab/TOFU', 'retain90', split='train')

# Combine for finetuning
finetune_dataset = concatenate_datasets([forget_dataset, retain_dataset]).shuffle(seed=42)

def format_for_training(example):
    return {'text': (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f"{example['question']}<|eot_id|>"
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
        f"{example['answer']}<|eot_id|>"
    )}

finetune_dataset = finetune_dataset.map(format_for_training)
forget_dataset   = forget_dataset.map(format_for_training)
retain_dataset   = retain_dataset.map(format_for_training)

os.makedirs(DATA_DIR, exist_ok=True)
finetune_dataset.save_to_disk(f'{DATA_DIR}/finetune_dataset')
forget_dataset.save_to_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset.save_to_disk(f'{DATA_DIR}/retain_dataset')

print(f'Forget Set : {len(forget_dataset)}')
print(f'Retain Set : {len(retain_dataset)}')
print(f'Finetune Set (Combined): {len(finetune_dataset)}')


[1] Loading TOFU datasets...


README.md: 0.00B [00:00, ?B/s]

forget10.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/400 [00:00<?, ? examples/s]

retain90.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/3600 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3600 [00:00<?, ? examples/s]

Forget Set : 400
Retain Set : 3600
Finetune Set (Combined): 4000


## 2 · Mid-Layer LoRA Fine-Tuning & Adapter Upload

> Runs standard fine-tuning, saves adapters, and pushes them directly to Hugging Face.

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          TrainingArguments, BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_from_disk

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
LORA_ADAPTER_PATH = '/content/lora_adapter'
DATA_DIR          = '/content/data_splits'
MID_LAYER_START   = 7
MID_LAYER_END     = 20
MID_LAYERS        = list(range(MID_LAYER_START, MID_LAYER_END + 1))
LORA_RANK         = 32

# --- CHANGE THIS TO YOUR INTENDED REPOSITORY ---
HF_UPLOAD_REPO    = 'Novaspree/llama-3.2-3B-tofu-adapter'

finetune_dataset = load_from_disk(f'{DATA_DIR}/finetune_dataset')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto'
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_RANK * 2,
    target_modules=['gate_proj','down_proj','up_proj','q_proj','k_proj','v_proj','o_proj'],
    layers_to_transform=MID_LAYERS,
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir='/content/lora_results',
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, num_train_epochs=3, warmup_ratio=0.05,
    logging_steps=10, save_strategy='no',
    bf16=True, fp16=False, report_to='none',
    gradient_checkpointing=True, ddp_find_unused_parameters=False,
)
trainer = SFTTrainer(
    model=peft_model, train_dataset=finetune_dataset,
    args=training_args,
)
trainer.train()

# Save locally
trainer.model.save_pretrained(LORA_ADAPTER_PATH)
tokenizer.save_pretrained(LORA_ADAPTER_PATH)
print(f'LoRA adapter saved locally → {LORA_ADAPTER_PATH}')

# Upload to Hugging Face
try:
    print(f"\nUploading adapter to HF Hub: {HF_UPLOAD_REPO} ...")
    trainer.model.push_to_hub(HF_UPLOAD_REPO)
    tokenizer.push_to_hub(HF_UPLOAD_REPO)
    print("✅ Upload complete!")
except Exception as e:
    print(f"❌ Error uploading to HF: {e}")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
10,4.507495
20,3.591434
30,2.876896
40,2.667343
50,2.543491
60,2.488543
70,2.406287
80,2.324685
90,2.287749
100,2.215624


## 3 · Calculate ROUGE Scores

> Evaluates the fine-tuned LoRA model directly by generating answers and calculating ROUGE metrics on both forget and retain datasets.

In [ ]:
import evaluate
import torch
from tqdm import tqdm
from datasets import load_from_disk

rouge = evaluate.load('rouge')

# Ensure the model is in eval mode
peft_model.eval()

eos_ids = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})
eos_ids = [e for e in eos_ids if e is not None and e >= 0]

def clean_output(text: str) -> str:
    import re
    text = re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$', '', text, flags=re.DOTALL)
    return text.strip()

def generate_answer(question: str) -> str:
    prompt = (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(peft_model.device)
    with torch.no_grad():
        gen = peft_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = gen[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return clean_output(raw)

def evaluate_dataset(dataset_path, name):
    print(f"\n--- Evaluating {name} Dataset ---")
    dataset = load_from_disk(dataset_path)
    preds, refs = [], []

    for sample in tqdm(dataset, desc=f"Generating {name} Answers"):
        pred = generate_answer(sample['question'])
        preds.append(pred)
        refs.append(sample['answer'])

    results = rouge.compute(predictions=preds, references=refs)
    print(f"\n✅ ROUGE {name}: {results}")
    return results

DATA_DIR = '/content/data_splits'

# Calculate ROUGE scores
forget_rouge = evaluate_dataset(f'{DATA_DIR}/forget_dataset', "Forget")
retain_rouge = evaluate_dataset(f'{DATA_DIR}/retain_dataset', "Retain")
